# Pipeline de Datos con Patrón Medallón

**Volumen fuente:** `/Volumes/proyecto_bda/bda_schema/bda_volumen`  
**Archivos:** `reviews_raw.csv` y `BASE_CULINARIA_SOLO_LOCALES_CON_RESEÑAS_2026.csv`

**Tablas Gold generadas:**
- `gold_reviews_final` — tabla maestra reseña + local + sentimiento
- `gold_stats_categoria` — estadísticas agregadas por categoría de restaurante
- `gold_features_usuario` — features por usuario para KMeans


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CATALOGO = "proyecto_bda"
SCHEMA   = "bda_schema"

#Rutas/ csv´s en el volumen
VOLUMEN_BASE = f"/Volumes/{CATALOGO}/{SCHEMA}/bda_volumen"
CSV_REVIEWS  = f"{VOLUMEN_BASE}/reviews_raw.csv"
CSV_PLACES   = f"{VOLUMEN_BASE}/BASE_CULINARIA_SOLO_LOCALES_CON_RESEÑAS_2026.csv"
#Tablas
TBL_BRONZE_REVIEWS    = f"{CATALOGO}.{SCHEMA}.bronze_reviews"
TBL_BRONZE_PLACES     = f"{CATALOGO}.{SCHEMA}.bronze_places"
TBL_SILVER_REVIEWS    = f"{CATALOGO}.{SCHEMA}.silver_reviews"
TBL_SILVER_PLACES     = f"{CATALOGO}.{SCHEMA}.silver_places"
# Gold: tres tablas con propósitos distintos
TBL_GOLD_FINAL        = f"{CATALOGO}.{SCHEMA}.gold_reviews_final"
TBL_GOLD_STATS_CAT    = f"{CATALOGO}.{SCHEMA}.gold_stats_categoria"
TBL_GOLD_FEAT_USUARIO = f"{CATALOGO}.{SCHEMA}.gold_features_usuario"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("Configuración lista.")
print(f"  Fuente reviews : {CSV_REVIEWS}")
print(f"  Fuente locales : {CSV_PLACES}")
print(f"  Tablas destino : {CATALOGO}.{SCHEMA}.*")


Configuración lista.
  Fuente reviews : /Volumes/proyecto_bda/bda_schema/bda_volumen/reviews_raw.csv
  Fuente locales : /Volumes/proyecto_bda/bda_schema/bda_volumen/BASE_CULINARIA_SOLO_LOCALES_CON_RESEÑAS_2026.csv
  Tablas destino : proyecto_bda.bda_schema.*



## 1. Capa BRONZE - Ingesta Raw
Lee los CSVs del volumen y los guarda como tablas delta

In [0]:
# Bronze: Reseñas ───────────────────────────────────────────
def ejecutar_bronze_reviews():
    df_bronze = spark.read.format("csv").option("header", "true").option("sep", ",").option("encoding", "UTF-8").load(CSV_REVIEWS)
    print(f"Columnas: {df_bronze.columns}")
    # overwriteSchema=True permite reejecutar sin errores de esquema
    df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_BRONZE_REVIEWS)

    count = spark.table(TBL_BRONZE_REVIEWS).count()
    print(f"Registros en Bronze (Reseñas): {count:,}")
    print(f"Tabla guardada: {TBL_BRONZE_REVIEWS}\n")

ejecutar_bronze_reviews()

Columnas: ['place_id', 'id_review', 'caption', 'relative_date', 'review_date', 'retrieval_date', 'rating', 'username', 'n_review_user', 'n_photo_user', 'url_user', 'url_source']
Registros en Bronze (Reseñas): 1,303,204
Tabla guardada: proyecto_bda.bda_schema.bronze_reviews



In [0]:
#Bronze: Locales ───────────────────────────────────────────
def ejecutar_bronze_places():
    df_bronze = spark.read.format("csv").option("header", "true").option("sep", ",").option("quote", "\"").option("escape", "\"").option("encoding", "UTF-8").option("multiLine", "true").load(CSV_PLACES)

    print(f"Columnas: {df_bronze.columns}")
    df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_BRONZE_PLACES)

    count = spark.table(TBL_BRONZE_PLACES).count()
    print(f"Registros en Bronze (Locales): {count:,}")
    print(f"Tabla guardada: {TBL_BRONZE_PLACES}\n")

ejecutar_bronze_places()

Columnas: ['id', 'url_place', 'title', 'category', 'address', 'phoneNumber', 'completePhoneNumber', 'domain', 'url', 'coor', 'stars', 'reviews', 'source_query']
Registros en Bronze (Locales): 39,947
Tabla guardada: proyecto_bda.bda_schema.bronze_places



## 2. Capa SILVER - Limpieza y Transformación
Aplica la misma logica de limpieza que "BDA_entrega_parcial":
deduplicación, casteos, texto limpio, feature engineering temporal y de texto.

In [0]:
#Silver: Reseñas
def ejecutar_silver_reviews():
    df = spark.table(TBL_BRONZE_REVIEWS)

    #casteos seguros con try_cast (toleran filas corrompidas)
    df_silver=df.withColumn("rating",F.expr("try_cast(rating AS DOUBLE)")).withColumn("review_date",F.expr("try_cast(review_date AS TIMESTAMP)"))

    nulos_rating = df_silver.filter(F.col("rating").isNull()).count() 
    nulos_fecha  = df_silver.filter(F.col("review_date").isNull()).count()
    print(f"  Filas con rating inválido   : {nulos_rating:,}")
    print(f"  Filas con fecha inválida    : {nulos_fecha:,}")

    df_silver = df_silver.fillna({"caption": "", "username": "Usuario Anónimo"})
    df_silver = df_silver.dropDuplicates(["id_review"])
    df_silver = df_silver.filter(F.col("rating").isNotNull() & F.col("review_date").isNotNull())

    #  Feature engineering textual 
    df_silver = df_silver.withColumn("text_len",F.length(F.col("caption"))).withColumn("word_count", F.size(F.split(F.trim(F.col("caption")), r"\s+")))

    df_silver = df_silver.withColumn("caption_clean",F.regexp_replace(F.lower(F.col("caption")), r"[^a-záéíóúñ0-9\s]", ""))
    df_silver = df_silver.withColumn("rating_category",F.when(F.col("rating") <= 2, "Baja").when(F.col("rating") == 3,"Media").otherwise("Alta"))
    #separo la fecha en años,meses, dias
    df_silver = df_silver.withColumn("year",F.year("review_date")).withColumn("month",F.month("review_date")).withColumn("day_of_week", F.dayofweek("review_date"))

    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_SILVER_REVIEWS)

    count = spark.table(TBL_SILVER_REVIEWS).count()
    print(f"Registros limpios en Silver (Reseñas): {count:,}")
    print(f"Tabla guardada: {TBL_SILVER_REVIEWS}\n")

ejecutar_silver_reviews()


  Filas con rating inválido   : 5,514
  Filas con fecha inválida    : 1,958
Registros limpios en Silver (Reseñas): 1,294,709
Tabla guardada: proyecto_bda.bda_schema.silver_reviews



In [0]:
#Silver: Locales 
def ejecutar_silver_places():
    df = spark.table(TBL_BRONZE_PLACES)
    #  Selección y renombrado de columnas 
    #    title -> name así se usa en BDA_entrega_parcial
    df_silver = df.select(F.col("id"),
        F.col("title").alias("name"),
        F.col("category"),
        F.col("address"),
        F.col("phoneNumber"),
        F.col("coor"),
        F.col("stars").cast(DoubleType()).alias("avg_rating"),
        F.when(F.col("url") == "", None).otherwise(F.col("url")).alias("url"),
        F.col("url_place")
    )

    # Extracción de lat/lng desde columna 'coor' ('lat, lng')
    #  Equivalente a df_places['coor'].str.split(',') del BDA
    df_silver = df_silver.withColumn("latitude",F.trim(F.split(F.col("coor"), ",").getItem(0)).cast(DoubleType())) \
        .withColumn("longitude",F.trim(F.split(F.col("coor"), ",").getItem(1)).cast(DoubleType()))

    #Deduplicación por id del local 
    df_silver = df_silver.dropDuplicates(["id"])
    # Relleno de nombre nulo 
    df_silver = df_silver.fillna({"name": "Restaurante Desconocido"})
    #  Descartamos locales sin coordenadas válidas
    df_silver = df_silver.filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())

    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_SILVER_PLACES)

    count = spark.table(TBL_SILVER_PLACES).count()
    print(f"Registros limpios en Silver (Locales): {count:,}")
    print(f"Tabla guardada: {TBL_SILVER_PLACES}\n")

ejecutar_silver_places()

Registros limpios en Silver (Locales): 39,945
Tabla guardada: proyecto_bda.bda_schema.silver_places



## 3. Capa GOLD - Agregaciones y Features para Modelos
>`gold_reviews_final` - tabla maestra (join reseñas + locales + sentimiento)
>`gold_stats_categoria` - estadísticas agregadas por categoría (para análisis descriptivo)
>`gold_features_usuario` - features por usuario listos para KMeans


In [0]:
# Gold: tabla maestra (join + sentimiento)
def ejecutar_gold_maestra():
    print("Tabla maestra")
    df_reviews = spark.table(TBL_SILVER_REVIEWS)
    df_places  = spark.table(TBL_SILVER_PLACES)

    df_gold = df_reviews.join(df_places,df_reviews.place_id == df_places.id,"left").drop(df_places.id)

    df_gold = df_gold.withColumn("sentimiento",
        F.when(F.col("rating") >= 4,"Positivo").when(F.col("rating") == 3,"Neutral").otherwise("Negativo"))

    df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_GOLD_FINAL)

    count = spark.table(TBL_GOLD_FINAL).count()
    print(f"  Registros: {count:,}")
    print(f"  Tabla: {TBL_GOLD_FINAL}\n")

    spark.table(TBL_GOLD_FINAL).createOrReplaceTempView("tabla_maestra_reviews")
    print("Vista temporal de: tabla_maestra_reviews registrada.")

ejecutar_gold_maestra()


Tabla maestra
  Registros: 1,294,709
  Tabla: proyecto_bda.bda_schema.gold_reviews_final

Vista temporal de: tabla_maestra_reviews registrada.


In [0]:
# Gold: estadísticas agregadas por categoría
def ejecutar_gold_stats_categoria():
    print("Estadísticas por categoría")
    df = spark.table(TBL_GOLD_FINAL)

    df_stats = df.groupBy("category").agg(
        F.count("id_review").alias("total_resenas"),
        F.round(F.avg("rating"), 2).alias("rating_promedio"),
        F.round(F.stddev("rating"), 2).alias("rating_desviacion"),
        F.sum(F.when(F.col("sentimiento") == "Positivo", 1).otherwise(0)).alias("resenas_positivas"),
        F.sum(F.when(F.col("sentimiento") == "Negativo", 1).otherwise(0)).alias("resenas_negativas"),
        F.round(F.avg("word_count"), 1).alias("promedio_palabras"),
        F.countDistinct("place_id").alias("total_locales")
    ).withColumn(
        "porcentaje_aprobacion",
        F.round(F.col("resenas_positivas") / F.col("total_resenas") * 100, 2)
    ).filter(F.col("category").isNotNull()) \
     .orderBy(F.desc("total_resenas"))

    df_stats.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_GOLD_STATS_CAT)

    count = spark.table(TBL_GOLD_STATS_CAT).count()
    print(f"  Categorías procesadas: {count:,}")
    print(f"  Tabla: {TBL_GOLD_STATS_CAT}\n")
    display(spark.table(TBL_GOLD_STATS_CAT).limit(10))

ejecutar_gold_stats_categoria()


Estadísticas por categoría
  Categorías procesadas: 107
  Tabla: proyecto_bda.bda_schema.gold_stats_categoria



category,total_resenas,rating_promedio,rating_desviacion,resenas_positivas,resenas_negativas,promedio_palabras,total_locales,porcentaje_aprobacion
Restaurante,669389,4.12,1.2,515271,74654,7.7,8370,76.98
Restaurante peruano,92924,4.23,1.16,74496,9136,9.4,792,80.17
Pizzería,83091,4.23,1.16,66778,8024,7.9,1094,80.37
Cafetería,56487,4.34,1.08,47344,4512,10.2,830,83.81
Restaurante especializado en pollo,47412,4.06,1.25,35713,5968,7.3,647,75.32
Restaurante de comida rápida,44694,4.08,1.26,33943,5622,6.9,1018,75.95
Restaurante chino,43510,3.97,1.25,31496,5822,7.1,419,72.39
Panadería,33075,4.21,1.13,26245,2961,7.0,940,79.35
Heladería,29865,4.37,1.07,25178,2251,8.0,801,84.31
Marisquería,25408,4.08,1.23,19228,3071,7.4,382,75.68


In [0]:
#Gold: features por usuario para KMeans despues
# Estas features son las que consume BDA_entrega_parcial
# para el clustering, reemplazando el groupby de pandas
def ejecutar_gold_features_usuario():
    print("Gold 3/3: Features por usuario (para KMeans)")
    df = spark.table(TBL_GOLD_FINAL)

    df_feat = df.groupBy("username").agg(
        F.count("id_review").alias("review_count"),
        F.round(F.avg("rating"), 4).alias("avg_rating"),
        F.round(F.stddev("rating"), 4).alias("std_rating"),
        F.round(F.avg("word_count"), 4).alias("avg_word_count"),
        # Transformación log para corregir asimetría (igual que en BDA pandas)
        F.round(F.log1p(F.count("id_review")), 4).alias("log_review_count"),
        F.round(F.log1p(F.avg("word_count")), 4).alias("log_avg_word_count")
    ).fillna({"std_rating": 0.0})  # usuarios con 1 sola reseña tienen std=null

    df_feat.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TBL_GOLD_FEAT_USUARIO)

    count = spark.table(TBL_GOLD_FEAT_USUARIO).count()
    print(f"  Usuarios procesados: {count:,}")
    print(f"  Tabla: {TBL_GOLD_FEAT_USUARIO}\n")
    display(spark.table(TBL_GOLD_FEAT_USUARIO).limit(5))

ejecutar_gold_features_usuario()


Gold 3/3: Features por usuario (para KMeans)
  Usuarios procesados: 711,324
  Tabla: proyecto_bda.bda_schema.gold_features_usuario



username,review_count,avg_rating,std_rating,avg_word_count,log_review_count,log_avg_word_count
Jesus Garcia Luna,1,5.0,0.0,1.0,0.6931,0.6931
Zuleika Gomez,1,5.0,0.0,18.0,0.6931,2.9444
Fernando Paiva Pingo,1,5.0,0.0,1.0,0.6931,0.6931
Alexandra Chavez,2,5.0,0.0,1.0,1.0986,0.6931
Milagros Segovia,1,5.0,0.0,1.0,0.6931,0.6931


## 4 Validación del Pipeline

In [0]:
tablas = {
    "Bronze Reviews":      TBL_BRONZE_REVIEWS,
    "Bronze Places":       TBL_BRONZE_PLACES,
    "Silver Reviews":      TBL_SILVER_REVIEWS,
    "Silver Places":       TBL_SILVER_PLACES,
    "Gold Maestra":        TBL_GOLD_FINAL,
    "Gold Stats Categoria": TBL_GOLD_STATS_CAT,
    "Gold Feat Usuario":   TBL_GOLD_FEAT_USUARIO,
}

print(f"{'Tabla':<25} {'Registros':>12} {'Estado':>8}")
print("=" * 50)
for nombre, tabla in tablas.items():
    try:
        n = spark.table(tabla).count()
        print(f"{nombre:<25} {n:>12,} {'OK':>8}")
    except Exception as e:
        print(f"{nombre:<25} {'—':>12} {'ERROR':>8}: {e}")


Tabla                        Registros   Estado
Bronze Reviews               1,303,204       OK
Bronze Places                   39,947       OK
Silver Reviews               1,294,709       OK
Silver Places                   39,945       OK
Gold Maestra                 1,294,709       OK
Gold Stats Categoria               107       OK
Gold Feat Usuario              711,324       OK


In [0]:
# Vista previa de la tabla Gold
display(spark.table(TBL_GOLD_FINAL).limit(10))

place_id,id_review,caption,relative_date,review_date,retrieval_date,rating,username,n_review_user,n_photo_user,url_user,url_source,text_len,word_count,caption_clean,rating_category,year,month,day_of_week,name,category,address,phoneNumber,coor,avg_rating,url,url_place,latitude,longitude,sentimiento
ChIJIwPlQ0RlEJERXa0t38g5RFo,ChZDSUhNMG9nS0VJQ0FnSUM3NnFxVkZ3EAE,,Hace un año,2025-04-25T11:11:50.095Z,2026-04-25 11:11:50.095934,5.0,Raquel Ronceros,0,null,https://www.google.com/maps/contrib/101124057391333864132/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJIwPlQ0RlEJERXa0t38g5RFo,0,1,,Alta,2025,4,6,Chifa Chang Sin San,Restaurante chino,Pisco 11601,907 088 388,"-13.734359399999999,-76.2239312",3.8,null,https://www.google.com/maps/place/?q=place_id:ChIJIwPlQ0RlEJERXa0t38g5RFo,-13.734359399999999,-76.2239312,Positivo
ChIJEbZHkeXHBZERoOVrBM0E6cE,ChZDSUhNMG9nS0VJQ0FnSUNQX3NDUUtnEAE,"Genial espectacular todo! A mi familia le encanta y a mi también, pasamos gratos momentos.",Hace un año,2025-04-25T11:12:23.423Z,2026-04-25 11:12:23.423344,5.0,Rosmery Zapata,0,null,https://www.google.com/maps/contrib/101735256005796312701/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJEbZHkeXHBZERoOVrBM0E6cE,90,15,genial espectacular todo a mi familia le encanta y a mi también pasamos gratos momentos,Alta,2025,4,6,El Pechugon,Restaurante peruano,"Av. Aviación 3870, Surquillo 15038",(01) 2337601,"-12.1139453,-76.9999476",4.3,null,https://www.google.com/maps/place/?q=place_id:ChIJEbZHkeXHBZERoOVrBM0E6cE,-12.1139453,-76.9999476,Positivo
ChIJX76WVAAbQA0RMA8s9x8nXaE,ChZDSUhNMG9nS0VJQ0FnSUNYbll2YUp3EAE,"Sitio muy recomendado para comer y cenar en Talavera. Toda la comida estaba riquísima, sobre todo la causa acevichada. El servicio muy amable y atento. Están ubicados en una placita y tienen una terraza amplia, por lo que tiene fácil acceso…",Hace un año,2025-04-25T11:12:46.889Z,2026-04-25 11:12:46.889684,5.0,J RH,0,null,https://www.google.com/maps/contrib/114311803785073586651/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJX76WVAAbQA0RMA8s9x8nXaE,241,41,sitio muy recomendado para comer y cenar en talavera toda la comida estaba riquísima sobre todo la causa acevichada el servicio muy amable y atento están ubicados en una placita y tienen una terraza amplia por lo que tiene fácil acceso,Alta,2025,4,6,El Gran Erizo,Restaurante peruano,"Calle J, 8, 45600 Talavera de la Reina, Toledo, España",610 82 26 22,"39.9643095,-4.8127745",4.6,http://wa.me/34610822622,https://www.google.com/maps/place/?q=place_id:ChIJX76WVAAbQA0RMA8s9x8nXaE,39.9643095,-4.8127745,Positivo
ChIJnZcpEghpXZERXHlOOr18l2g,ChdDSUhNMG9nS0VJQ0FnSUNKaGVha3dnRRAB,Probé un riquísimo chairo...recomendadisimo,Hace 2 años,2024-04-25T11:12:58.181Z,2026-04-25 11:12:58.181150,5.0,EMELISSA BROWN ALVAREZ,280,null,https://www.google.com/maps/contrib/103987470183397358109/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJnZcpEghpXZERXHlOOr18l2g,43,4,probé un riquísimo chairorecomendadisimo,Alta,2024,4,5,Restaurant Sombreritos,Restaurante,"Sesquicentenario 1102, Puno 21001",979 288 417,"-15.823579599999999,-70.0015418",3.7,null,https://www.google.com/maps/place/?q=place_id:ChIJnZcpEghpXZERXHlOOr18l2g,-15.823579599999999,-70.0015418,Positivo
ChIJEbZHkeXHBZERoOVrBM0E6cE,ChZDSUhNMG9nS0VJQ0FnSUNqXzVXWkN3EAE,,Hace un año,2025-04-25T11:13:03.687Z,2026-04-25 11:13:03.687955,5.0,Jéssica Valenzuela Díaz,106,null,https://www.google.com/maps/contrib/111084136385756713774/reviews?hl=es,https://www.google.com/maps/place/?q=place_id:ChIJEbZHkeXHBZERoOVrBM0E6cE,0,1,,Alta,2025,4,6,El Pechugon,Restaurante peruano,"Av. Aviación 3870, Surquillo 15038",(01) 2337601,"-12.1139453,-76.9999476",4.3,null,https://www.google.com/maps/place/?q=place_id:ChIJEbZHkeXHBZERoOVrBM0E6cE,-12.1139453,-76.9999476,Positivo
ChIJEbZHkeXHBZERoOVrBM0E6cE,ChdDSUhNMG9nS0VJQ0FnSUNqbjlTN2lnRRAB,,Hace un año,2025-04-25T11:13:03.687Z,2026-04-25 11:13:03.687955,3.0,Eduardo Voysest Gamarra,0,null,https:

## 5 Insights SQL sobre la Capa Gold

In [0]:
# Recarga de vista por si se reinicia el notebook
spark.table(TBL_GOLD_FINAL).createOrReplaceTempView("tabla_maestra_reviews")
print("Vista 'tabla_maestra_reviews' registrada")

In [0]:
# Insight 1: Ranking de Satisfacción por Categoría 
spark.sql("""
    SELECT
        category                                                          AS Categoria,
        COUNT(id_review)                                                  AS Total_Resenas,
        ROUND(AVG(rating), 2)                                             AS Calificacion_Promedio,
        SUM(CASE WHEN sentimiento = 'Positivo' THEN 1 ELSE 0 END)        AS Resenas_Positivas,
        SUM(CASE WHEN sentimiento = 'Negativo' THEN 1 ELSE 0 END)        AS Quejas,
        ROUND(
            SUM(CASE WHEN sentimiento = 'Positivo' THEN 1 ELSE 0 END)
            / COUNT(id_review) * 100, 2
        )                                                                 AS Porcentaje_Aprobacion
    FROM tabla_maestra_reviews
    WHERE category IS NOT NULL
    GROUP BY category
    HAVING Total_Resenas > 500
    ORDER BY Porcentaje_Aprobacion DESC
    LIMIT 8
""").show(truncate=False)

+------------------------------------+-------------+---------------------+-----------------+------+---------------------+
|Categoria                           |Total_Resenas|Calificacion_Promedio|Resenas_Positivas|Quejas|Porcentaje_Aprobacion|
+------------------------------------+-------------+---------------------+-----------------+------+---------------------+
|Restaurante orgánico                |720          |4.9                  |702              |9     |97.5                 |
|Restaurante americano               |1698         |4.61                 |1553             |81    |91.46                |
|Restaurante vegano                  |4000         |4.59                 |3585             |219   |89.63                |
|Restaurante de cocina española      |664          |4.55                 |595              |28    |89.61                |
|Restaurante de alta cocina          |538          |4.51                 |474              |32    |88.1                 |
|Restaurante italiano   

In [0]:
# Insight 2: Tracción de Grandes Cadenas
spark.sql("""
    SELECT
        name                          AS Nombre_Restaurante,
        COUNT(DISTINCT place_id)      AS Sucursales_Registradas,
        COUNT(id_review)              AS Total_Interacciones,
        ROUND(AVG(rating), 2)         AS Calificacion_Global,
        MAX(category)                 AS Categoria_Principal
    FROM tabla_maestra_reviews
    WHERE name != 'Restaurante Desconocido'
      AND name IS NOT NULL
    GROUP BY name
    HAVING Total_Interacciones > 1000
    ORDER BY Total_Interacciones DESC
    LIMIT 20
""").show(truncate=False)

+------------------------------+----------------------+-------------------+-------------------+----------------------------------+
|Nombre_Restaurante            |Sucursales_Registradas|Total_Interacciones|Calificacion_Global|Categoria_Principal               |
+------------------------------+----------------------+-------------------+-------------------+----------------------------------+
|KFC                           |38                    |8272               |3.86               |Restaurante de comida rápida      |
|Bembos                        |14                    |2307               |3.93               |Restaurante de comida rápida      |
|Corralito                     |7                     |1881               |4.14               |Restaurante peruano               |
|McDonald's                    |10                    |1703               |3.87               |Restaurante de comida rápida      |
|Roky's                        |7                     |1616               |3.91    

In [0]:
# Insight 3: Engagement por Categoría de Rating
spark.sql("""
    SELECT
        rating_category            AS Categoria_Rating,
        COUNT(id_review)           AS Total_Reviews,
        ROUND(AVG(text_len), 1)    AS Promedio_Chars,
        ROUND(AVG(word_count), 1)  AS Promedio_Palabras
    FROM tabla_maestra_reviews
    GROUP BY rating_category
    ORDER BY Total_Reviews DESC
""").show()

+----------------+-------------+--------------+-----------------+
|Categoria_Rating|Total_Reviews|Promedio_Chars|Promedio_Palabras|
+----------------+-------------+--------------+-----------------+
|            Alta|      1012935|          38.5|              7.0|
|           Media|       143349|          39.8|              7.6|
|            Baja|       138425|          88.2|             16.3|
+----------------+-------------+--------------+-----------------+

